# SME Financial Health Index Modeling Pipeline

This notebook presents the full reproducible end-to-end workflow for predicting SME Financial Health Index (`Low`, `Medium`, `High`) from the Zindi challenge data.

The workflow covers:
- data loading
- data review and class balance inspection
- feature engineering
- multiclass model training
- F1-focused ensemble optimization
- evaluation and subgroup analysis
- submission generation


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
PROJECT_ROOT


In [ ]:
import sys
sys.path.append(str(PROJECT_ROOT))

from src.data.loaders import load_train_data, load_test_data, load_variable_definitions
from src.features.engineering import engineer_features
from src.models.training import train_pipeline
from src.inference.predict import load_artifact, predict_dataframe


## Load the raw data

In [ ]:
train_df = load_train_data()
test_df = load_test_data()
variable_definitions = load_variable_definitions()

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Variable definition rows:', variable_definitions.shape[0])
train_df.head()


## Review the target balance and missingness

In [ ]:
target_distribution = train_df['Target'].value_counts().rename_axis('Target').to_frame('count')
target_distribution['share'] = target_distribution['count'] / len(train_df)
missing_summary = train_df.isna().mean().sort_values(ascending=False).head(15).rename('missing_share')

print('Target distribution')
display(target_distribution)
print('Top missing features')
display(missing_summary.to_frame())


## Country-level target patterns

In [ ]:
country_target = pd.crosstab(train_df['country'], train_df['Target'], normalize='index').round(3)
country_target


## Feature engineering preview

In [ ]:
feature_frame = engineer_features(train_df)
print('Engineered feature shape:', feature_frame.shape)
feature_frame.head()


## Train the optimized pipeline

This cell runs the reproducible training pipeline implemented in `src/models/training.py`. It benchmarks multiple strong tabular models, generates out-of-fold probabilities, evaluates weighted F1 and macro F1, and searches for an improved probability-weighted ensemble.

In [ ]:
training_outputs = train_pipeline(train_df)
training_outputs['selection']


## Compare model performance

In [ ]:
model_comparison = pd.read_csv(PROJECT_ROOT / 'outputs' / 'metrics' / 'model_comparison.csv')
model_comparison


## Inspect the best model selection details

In [ ]:
selection = json.loads((PROJECT_ROOT / 'outputs' / 'metrics' / 'model_selection.json').read_text())
selection


## Review the classification report

In [ ]:
classification_report = json.loads((PROJECT_ROOT / 'outputs' / 'metrics' / 'classification_report.json').read_text())
pd.DataFrame(classification_report).T


## Review subgroup performance

In [ ]:
subgroup_analysis = pd.read_csv(PROJECT_ROOT / 'outputs' / 'metrics' / 'subgroup_analysis.csv')
subgroup_analysis.sort_values(['weighted_f1', 'macro_f1']).head(15)


## Load the saved artifact and score the test set

In [ ]:
artifact = load_artifact(PROJECT_ROOT / 'artifacts' / 'trained_pipeline.joblib')
test_predictions = predict_dataframe(test_df, artifact)
test_predictions.head()


## Save the final submission file

In [ ]:
submission_path = PROJECT_ROOT / 'outputs' / 'submissions' / 'submission_B.csv'
test_predictions[['ID', 'Target']].to_csv(submission_path, index=False)
print(submission_path)
print(pd.read_csv(submission_path).head().to_string(index=False))


## Geography-aware experiment

This section reviews the leakage-safe country specialist experiment. Country-specific models were trained only inside training folds, with a weighted fallback for sparse geographies such as Lesotho.

In [ ]:
model_comparison = pd.read_csv(PROJECT_ROOT / 'outputs' / 'metrics' / 'model_comparison.csv')
model_comparison[model_comparison['model'].isin(['weighted_ensemble_optimized', 'geo_weighted_blend', 'geo_hybrid_stacker'])]

In [ ]:
selection = json.loads((PROJECT_ROOT / 'outputs' / 'metrics' / 'model_selection.json').read_text())
{
    'country_specialist_model_name': selection['country_specialist_model_name'],
    'country_strategy_counter': selection['country_strategy_counter'],
    'geo_weighted_blend_summary': selection['geo_weighted_blend_summary'],
    'geo_hybrid_stacker_summary': selection['geo_hybrid_stacker_summary'],
}

The geography-aware specialist blend improved `High`-class F1 slightly, but it still did not beat the best global optimized ensemble on weighted F1. For that reason, the final submission continues to use the stronger global optimized ensemble.

## Notes

- The best final model is a probability-weighted ensemble chosen by out-of-fold weighted F1.
- The final blend uses deterministic model weights and class-probability adjustments, so inference remains fully reproducible.
- The submission file is saved in the exact `ID,Target` format required by the competition template.

## High-Focused Experiments

This section summarizes the later non-leaky experiments focused on improving the rare `High` class without using geographic splitting as the main strategy.

In [ ]:
model_comparison = pd.read_csv(PROJECT_ROOT / 'outputs' / 'metrics' / 'model_comparison.csv')
model_comparison[model_comparison['model'].isin(['weighted_ensemble_optimized', 'binary_guided_ensemble', 'geo_weighted_blend'])][['model','weighted_f1','macro_f1','high_f1','medium_f1']]

The binary-guided ensemble improved `High` F1 and macro F1 relative to several baselines, but it still did not beat the best weighted-F1 model. Because the competition priority remains overall F1, the final recommended submission stays with the stronger weighted ensemble.